# Projeções em Computação Gráfica

Após as **transformações geométricas 3D** (translação, rotação, escala etc.), os objetos da cena estão definidos no **espaço do mundo**.  
Para exibi-los em uma tela 2D, é necessário projetá-los em um **plano de projeção**, reduzindo suas coordenadas tridimensionais $(x, y, z)$ para bidimensionais $(x', y')$.

Essa conversão é chamada de **transformação de projeção** e define o modo como o observador “enxerga” a cena.  
Existem dois tipos principais de projeção: **ortográfica (ou paralela)** e **em perspectiva**.

---

## 1. Projeção Ortográfica (ou Paralela)

### Conceito
Na **projeção ortográfica**, todas as linhas projetantes são **paralelas entre si** e **perpendiculares ao plano de projeção**.  
Isso significa que **não há redução aparente de tamanho com a distância**: objetos próximos e distantes do observador mantêm a mesma escala.

É amplamente usada em **desenhos técnicos, engenharia e CAD**, pois **preserva as proporções e ângulos**, sem distorções de perspectiva.

### Características principais
- Paralelismo e proporções são preservados  
- Linhas paralelas permanecem paralelas após a projeção  
- O observador é considerado “infinitamente distante”  
- Não há profundidade visual (sem ponto de fuga)  
- Adequada para vistas ortogonais (frontal, lateral, superior)

### Equação da projeção ortográfica

A projeção consiste em eliminar o componente de profundidade $(z)$:

$$
x' = x \\
y' = y
$$

No pipeline gráfico, essa projeção é representada por uma matriz $4 \times 4$ que transforma o volume visível (caixa ortográfica):

$$
P_{ortho} =
\begin{bmatrix}
\frac{2}{r - l} & 0 & 0 & -\frac{r + l}{r - l} \\
0 & \frac{2}{t - b} & 0 & -\frac{t + b}{t - b} \\
0 & 0 & -\frac{2}{f - n} & -\frac{f + n}{f - n} \\
0 & 0 & 0 & 1
\end{bmatrix}
$$

onde:
- $l, r$: limites esquerdo e direito  
- $b, t$: limites inferior e superior  
- $n, f$: planos de recorte (near e far)

O resultado é um **volume de visualização retangular (caixa)**, também chamado de **view volume ortográfico**.

---

## 2. Projeção em Perspectiva

### Conceito
Na **projeção em perspectiva**, as linhas projetantes **convergem para um ponto de fuga** (geralmente o olho do observador).  
Dessa forma, **objetos mais distantes parecem menores**, simulando a percepção de profundidade do olho humano.

É a projeção utilizada em **renderizações realistas, jogos, simulações e animações**.

### Características principais
- Mantém a sensação de profundidade (efeito de “distância”)  
- Linhas paralelas **convergem** para um ponto de fuga  
- Objetos distantes sofrem redução aparente de tamanho  
- A escala **não é preservada**, mas a percepção é natural  

### Equação da projeção perspectiva

A projeção é obtida considerando o **triângulo de semelhança** entre o ponto e o plano de projeção:

$$
x' = \frac{x \cdot d}{z} \\
y' = \frac{y \cdot d}{z}
$$

onde:
- $d$ é a distância do plano de projeção à origem  
- O fator $\frac{1}{z}$ causa o efeito de diminuição com a profundidade

No pipeline, é usada uma matriz $4 \times 4$:

$$
P_{persp} =
\begin{bmatrix}
\frac{n}{r} & 0 & 0 & 0 \\
0 & \frac{n}{t} & 0 & 0 \\
0 & 0 & -\frac{f + n}{f - n} & -\frac{2fn}{f - n} \\
0 & 0 & -1 & 0
\end{bmatrix}
$$

Ou, de forma mais genérica, com o **campo de visão (FOV)** e proporção (aspect):

$$
P_{persp} =
\begin{bmatrix}
\frac{1}{\tan(\frac{fov}{2}) \cdot aspect} & 0 & 0 & 0 \\
0 & \frac{1}{\tan(\frac{fov}{2})} & 0 & 0 \\
0 & 0 & -\frac{f + n}{f - n} & -\frac{2fn}{f - n} \\
0 & 0 & -1 & 0
\end{bmatrix}
$$

---

## 3. Comparação entre Ortográfica e Perspectiva

| Aspecto | Projeção Ortográfica | Projeção em Perspectiva |
|----------|----------------------|--------------------------|
| Tipo de linhas projetantes | Paralelas | Convergentes |
| Ponto de fuga | Inexistente | Presente |
| Escala com distância | Constante | Diminui com o aumento da distância |
| Realismo | Baixo | Alto |
| Preservação de medidas | Sim | Não |
| Uso comum | Engenharia, CAD, arquitetura | Simulação, jogos, renderização 3D |
| Volume de visualização | Caixa (ortho box) | Pirâmide truncada (frustum) |

---

## 4. Relação com o Pipeline Gráfico

No pipeline gráfico moderno, a transformação de projeção faz parte da sequência:

$$
M_{final} = M_{projecao} \cdot M_{view} \cdot M_{model}
$$

1. **Model** – transforma o objeto para o espaço do mundo  
2. **View** – posiciona a câmera  
3. **Projection** – projeta o mundo 3D no plano da tela (ortho ou perspectiva)  
4. **Viewport** – mapeia o resultado para as coordenadas de pixel da tela




---



# Transformações de Visualização (Câmera / View Transformation)

Após a etapa de **modelagem** (transformações geométricas) e antes da **projeção**, é necessário definir o ponto de vista do observador.  
Essa etapa é chamada de **transformação de visualização** ou **transformação da câmera**, e tem como objetivo converter as coordenadas do **espaço do mundo** para o **espaço da câmera (ou espaço de visão)**.

---

## 1. Conceito

A transformação de visualização reposiciona e reorienta toda a cena de modo que:
- A **câmera (observador)** fique na origem $(0, 0, 0)$  
- O **eixo -Z** aponte na direção da visão  
- O **eixo +Y** aponte “para cima”  

Dessa forma, a câmera é tratada como estacionária, e todo o mundo é transformado em torno dela.  
Isso simplifica o cálculo da projeção e da renderização.

---

## 2. Sistema de Coordenadas da Câmera

O sistema de coordenadas da câmera é definido a partir de três vetores ortogonais:

- **n (forward)**: direção da visão (do olho para o alvo)  
- **u (right)**: eixo horizontal, obtido pelo produto vetorial entre o vetor “up” e o “n”  
- **v (up)**: eixo vertical da câmera, ortogonal aos demais

Esses vetores formam uma **base ortonormal** do sistema de coordenadas da câmera.

$$
\text{Dados:} \\
\text{eye} = (x_e, y_e, z_e) \quad \text{(posição do observador)} \\
\text{center} = (x_c, y_c, z_c) \quad \text{(ponto observado)} \\
\text{up} = (x_u, y_u, z_u) \quad \text{(vetor "para cima")}
$$

Calculamos os vetores da câmera como:

$$
n = \frac{eye - center}{\| eye - center \|} \\
u = \frac{up \times n}{\| up \times n \|} \\
v = n \times u
$$

onde $\times$ representa o **produto vetorial**.

---

## 3. Matriz de Visualização (LookAt)

A transformação de visualização pode ser representada por uma **matriz 4×4** que combina rotação e translação.  
Essa matriz é responsável por alinhar o sistema do mundo com o sistema da câmera.

A forma geral da matriz é:

$$
M_{view} =
\begin{bmatrix}
u_x & u_y & u_z & -u \cdot eye \\
v_x & v_y & v_z & -v \cdot eye \\
n_x & n_y & n_z & -n \cdot eye \\
0 & 0 & 0 & 1
\end{bmatrix}
$$

onde:
- $(u_x, u_y, u_z)$, $(v_x, v_y, v_z)$ e $(n_x, n_y, n_z)$ são os componentes dos vetores da base da câmera  
- $u \cdot eye$, $v \cdot eye$ e $n \cdot eye$ representam o **produto escalar** com o vetor posição da câmera  

Essa matriz realiza:
1. Uma **rotação** para alinhar os eixos da câmera com os eixos do mundo  
2. Uma **translação** para levar o observador até a origem

---

## 4. Interpretação Geométrica

A operação realizada pela matriz $M_{view}$ transforma todos os pontos da cena de acordo com a posição e orientação da câmera:

$$
p' = M_{view} \cdot p
$$

Após essa transformação:
- A câmera é colocada na origem $(0, 0, 0)$  
- O ponto de observação está ao longo do eixo **-Z**  
- O plano de projeção passa pela origem  

Essa convenção é adotada pelos sistemas gráficos modernos (OpenGL, DirectX, Vulkan).

---

## 5. Relação com a Função `gluLookAt()`

Em implementações clássicas do OpenGL, a função `gluLookAt()` constrói automaticamente essa matriz a partir dos parâmetros:

```c
gluLookAt(
    eyeX, eyeY, eyeZ,      // posição da câmera
    centerX, centerY, centerZ,  // ponto observado
    upX, upY, upZ           // vetor "para cima"
);




---



# Teoria do Viewport em Computação Gráfica e OpenGL

## Conceito Geral

O **viewport** é a etapa final do pipeline de renderização em computação gráfica.  
Ele define a **região da janela** (em coordenadas de tela) onde a imagem final será desenhada.  
Em outras palavras, o viewport realiza o **mapeamento das coordenadas normalizadas do dispositivo (NDC)** para as **coordenadas reais da tela (pixels)**.

Após as transformações **Model**, **View** e **Projection**, os vértices do objeto são convertidos para o espaço **NDC**, no qual cada coordenada varia entre -1 e 1:

$$
-1 \leq x_{ndc} \leq 1, \quad -1 \leq y_{ndc} \leq 1, \quad -1 \leq z_{ndc} \leq 1
$$

Essas coordenadas ainda não estão em pixels da tela; é o **viewport** que faz esse mapeamento.

---

## Mapeamento para o Viewport

O objetivo da transformação de viewport é converter o ponto $(x_{ndc}, y_{ndc})$ do espaço normalizado em coordenadas de janela $(x_w, y_w)$, considerando a largura e altura da janela.

A equação geral é:

$$
x_w = \frac{(x_{ndc} + 1)}{2} \cdot w + x_{min}
$$

$$
y_w = \frac{(y_{ndc} + 1)}{2} \cdot h + y_{min}
$$

onde:

- $w$ é a **largura do viewport** (em pixels)  
- $h$ é a **altura do viewport** (em pixels)  
- $(x_{min}, y_{min})$ é a **origem do viewport** (geralmente o canto inferior esquerdo da janela)  

O eixo $z$ também pode ser reescalado para o intervalo de profundidade definido em OpenGL:

$$
z_w = \frac{(z_{ndc} + 1)}{2} \cdot (z_{max} - z_{min}) + z_{min}
$$

Normalmente, o OpenGL usa $z_{min} = 0$ e $z_{max} = 1$.

---

## Implementação no OpenGL

No OpenGL, o viewport é definido pelo comando:

```c
glViewport(x, y, width, height);

onde:

x e y são as coordenadas da origem do viewport na janela (em pixels);

width e height definem a dimensão do viewport.



---



# Teoria de Iluminação em Computação Gráfica e OpenGL

## 1. Objetivo da iluminação
Iluminação em computação gráfica modela como a energia luminosa interage com superfícies e chega ao observador. O propósito é estimar a radiância de saída na direção da câmera a partir das propriedades de luz, material e geometria local.

A formulação integral geral é a **equação de renderização**:
$$
L_o(\mathbf{x},\omega_o) = L_e(\mathbf{x},\omega_o) + \int_{\Omega^+} f_r(\mathbf{x},\omega_i,\omega_o)\,L_i(\mathbf{x},\omega_i)\,(\mathbf{n}\cdot\mathbf{\omega}_i)\,d\omega_i
$$
em que $f_r$ é a BRDF, $L_i$ a radiância incidente, $\mathbf{n}$ a normal e $\Omega^+$ o hemisfério acima da superfície.

Para **iluminação local** com luzes pontuais e sem inter-reflexões, usa-se uma soma discreta:
$$
L_o \approx \sum_{k\in\text{luzes}} f_r(\omega_i^k,\omega_o)\,I_k\,\max(0,\mathbf{n}\cdot\mathbf{l}_k)
$$
com $\mathbf{l}_k$ direção da luz $k$ e $I_k$ sua intensidade.

---

## 2. Componentes clássicos de iluminação local
O modelo clássico de Phong separa a refletância em termos **ambiente**, **difuso** e **especular**.

- **Ambiente**: aproxima a iluminação indireta como constante
  $$
  C_{\text{amb}} = k_a \, I_a
  $$
  com $k_a$ coeficiente ambiente do material e $I_a$ intensidade ambiente da cena.

- **Difuso lambertiano**: dispersão uniforme da luz refletida
  $$
  C_{\text{dif}} = k_d\,I_k\,\max(0,\mathbf{n}\cdot\mathbf{l})
  $$
  em que $k_d$ é a cor difusa do material e $\mathbf{l}$ é a direção luz-superfície normalizada.

- **Especular**: brilho dependente da direção do observador
  - **Phong clássico** via vetor de reflexão $\mathbf{r}$:
    $$
    C_{\text{esp}} = k_s\,I_k\,\max(0,\mathbf{r}\cdot\mathbf{v})^{\alpha}
    $$
    com $\mathbf{r} = 2(\mathbf{n}\cdot\mathbf{l})\mathbf{n} - \mathbf{l}$, $\mathbf{v}$ direção superfície-observador, $k_s$ cor especular e $\alpha$ expoente de brilho.
  - **Blinn–Phong** via half-vector $\mathbf{h}$:
    $$
    C_{\text{esp}} = k_s\,I_k\,\max(0,\mathbf{n}\cdot\mathbf{h})^{\beta},\quad \mathbf{h}=\frac{\mathbf{l}+\mathbf{v}}{\|\mathbf{l}+\mathbf{v}\|}
    $$

A **cor final local** por luz é
$$
C = C_{\text{amb}} + C_{\text{dif}} + C_{\text{esp}}
$$
e acumula-se sobre todas as luzes.

---

## 3. Atenuação e tipos de luz
Para **luz pontual** com atenuação por distância $d$:
$$
\text{att}(d) = \frac{1}{k_c + k_l d + k_q d^2}
$$
A contribuição difusa/especular recebe o fator $\text{att}(d)$.

Para **spotlight** com direção $\mathbf{s}$ e ângulo de corte $\theta_c$, define-se
$$
\gamma = \cos^{-1}\!\left(\frac{\mathbf{s}\cdot(-\mathbf{l})}{\|\mathbf{s}\|}\right)
$$
e aplica-se um feixe suave
$$
\text{spot}(\gamma) =
\begin{cases}
(\cos\gamma)^{p} & \text{se } \gamma \le \theta_c \\
0 & \text{caso contrário}
\end{cases}
$$
O termo final por luz fica $\text{att}(d)\,\text{spot}(\gamma)\,(C_{\text{dif}}+C_{\text{esp}})$.

---

## 4. Normais, espaço e correções
- Todos os vetores $\mathbf{n}$, $\mathbf{l}$, $\mathbf{v}$, $\mathbf{r}$, $\mathbf{h}$ devem ser **normalizados**.
- Em pipeline clássico, normais são transformadas pelo inverso transposto da matriz model-view para preservar ortogonalidade.
- **Normal mapping** altera $\mathbf{n}$ por um mapa de normais definido no espaço de tangentes. Construção da base $\{\mathbf{t},\mathbf{b},\mathbf{n}\}$ e transformação do vetor normal do mapa para o espaço do mundo ou da view são necessárias.

---

## 5. Gouraud vs Phong shading
- **Gouraud shading**: computa iluminação por vértice e interpola cores na rasterização. Custo baixo e pode perder highlights pequenos.
- **Phong shading**: interpola normais por fragmento e computa iluminação por pixel. Preserva highlights e detalhes especulares.

---

## 6. Relação com OpenGL clássico
No pipeline fixo do OpenGL legado, habilita-se e configura-se iluminação via estado:

```c
glEnable(GL_LIGHTING);
glEnable(GL_LIGHT0); /* ... GL_LIGHT1, etc. */
GLfloat Ia[4] = {0.1f, 0.1f, 0.1f, 1.0f};
glLightModelfv(GL_LIGHT_MODEL_AMBIENT, Ia);

GLfloat L0_pos[4] = {5.0f, 8.0f, 10.0f, 1.0f};  /* pontual se w=1 */
GLfloat L0_diff[4] = {1,1,1,1};
GLfloat L0_spec[4] = {1,1,1,1};
glLightfv(GL_LIGHT0, GL_POSITION, L0_pos);
glLightfv(GL_LIGHT0, GL_DIFFUSE,  L0_diff);
glLightfv(GL_LIGHT0, GL_SPECULAR, L0_spec);

/* Atenuação */
glLightf(GL_LIGHT0, GL_CONSTANT_ATTENUATION,  1.0f);
glLightf(GL_LIGHT0, GL_LINEAR_ATTENUATION,    0.09f);
glLightf(GL_LIGHT0, GL_QUADRATIC_ATTENUATION, 0.032f);

/* Material */
glEnable(GL_COLOR_MATERIAL);
glColorMaterial(GL_FRONT_AND_BACK, GL_AMBIENT_AND_DIFFUSE);
GLfloat ks[4] = {0.8f,0.8f,0.8f,1.0f};
glMaterialfv(GL_FRONT_AND_BACK, GL_SPECULAR, ks);
glMaterialf (GL_FRONT_AND_BACK, GL_SHININESS, 64.0f);
